# Six-bit periodic quantum walk: native compilation workflow

This Notebook preserves the source's six position bits, coin, conditional
increment/decrement, maximum width of twelve and CX/U hardware basis objective.
It compiles two semantically equivalent high-level programs, converts both to
CX/U3, reports actual width, depth and gate counts, and selects the visited
candidate with fewer CX gates (ties: depth, width, name).

U3 is FlagQuantum's name for the three-angle U gate. The explicit, immutable
conversion-rule extension preserves absolute phase and leaves compiler defaults
unchanged. This is bounded selection between two candidates, not a claim of
globally optimal compilation or equivalence to a private optimizer.


In [ ]:
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import H, Output, QBit, QNum, allocate, control, qfunc


@qfunc
def increment(x: QNum) -> None:
    x += 1


@qfunc
def decrement(x: QNum) -> None:
    x -= 1


@qfunc
def walk(x: Output[Annotated[QNum, 6]], coin: Output[QBit]) -> None:
    allocate(x)
    allocate(coin)
    H(coin)
    control(coin == 0, lambda: increment(x))
    control(coin == 1, lambda: decrement(x))


program = synth.synthesize(walk, max_qubits=12, max_workspace=5)
result = fq.run(program)
print(dict(program.probabilities(result, registers=("x", "coin"))))


The source state places equal probability at (1, 0) and (63, 1).
The comparison also checks every logical input column and coherent inputs;
probabilities alone do not certify phase or clean work qubits.


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.classiq_discrete_quantum_walk.overview.workflow import (
    run_workflow,
)

workflow = run_workflow()
for candidate in workflow.candidates:
    cost = candidate.resources
    print(
        candidate.name,
        "physical_qubits",
        cost.n_qubits,
        "depth",
        cost.depth,
        "CX",
        cost.operation_counts.get("cx", 0),
        "U3",
        cost.operation_counts.get("u3", 0),
        "synthesis_seconds",
        candidate.synthesis_seconds,
        "conversion_seconds",
        candidate.conversion_seconds,
    )
print("selected", workflow.selected.name)
print("local_workflow_seconds", workflow.elapsed_seconds)


In [ ]:
selected = workflow.selected
selected_result = fq.run(selected.program)
print(dict(selected.program.probabilities(selected_result, registers=("x", "coin"))))


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.classiq_discrete_quantum_walk.overview.comparisons.circuit import (
    main as compare_circuit,
)

compare_circuit()


The measured Fourier candidate has 7 physical qubits, no work qubits,
88 CX and 124 U3 operations, depth 106. The arithmetic candidate has
10 physical qubits, 3 clean work qubits, 254 CX and 380 U3 operations,
depth 400. Counts refer to the actually converted CX/U3 IR, not native gates.
Timings are observed local durations for synthesis, conversion and the entire
workflow (including verification); they are not performance or hardware claims.
The source has no plot or sampling stage. No external SDK or remote job is used.
